In [ ]:
# 更新到最新版本
%pip install nextrec --quiet --upgrade

In [ ]:
pip show nextrec

# 5-Minute Quick Start

该Notebook将为大家了解 NextRec：一个统一、高效、可扩展的推荐系统框架，并带大家从零到一训练并构建一个可上线的推荐模型。示例数据集来自电商场景。

先介绍一些推荐系统的概念。在推荐系统中，通常会处理多种类型的输入信号，在经过一系列的变换之后转化为向量输入网络：

- 稠密特征（数值型）：连续或可序数化的数值，如年龄、价格、时长、打分；常见做法是标准化/归一化或对数变换。
- 稀疏特征（类别/ID）：高基数离散字段，如用户 ID、物品 ID、性别、职业、设备类型；通常需要索引化后，在一个embedding lookup matrix中进行嵌入。
- 序列特征（行为序列）：可变长的历史行为，如用户的浏览/点击/购买列表。这类特征表征了用户的行为和兴趣变化，通常我们需要截断、padding，嵌入后通过不同聚合方式（如 mean/sum/attention）将其变为定长向量。
- 上下文特征：时间、地理、曝光位置等环境信息，可是稠密也可能是稀疏，常与主特征交互。
- 多模态特征：文本、图片、视频等经过预训练模型得到的向量，可直接作为稠密输入，或与 ID 交互建模。

通常一个标准的训练数据格式如下所示：

```text
user_id,item_id,gender,age,occupation,history_seq,label
1024,501,1,28,3,"[12,45,18,77]",1
2048,777,0,35,5,"[8,99]",0
```

这里，我们提供了一份脱敏后的数据集供大家使用，该数据集来源于电商场景下，包含用户id，物品id，稠密特征，稀疏特征和序列特征，标签则包含是否点击，与是否转化。

In [ ]:
import pandas as pd

df = pd.read_csv('https://raw.githubusercontent.com/zerolovesea/NextRec/main/dataset/multitask_task.csv')
df.head()

In [ ]:
# 目标变量是点击和转化
task_labels = ['click', 'conversion']

# 根据列名将特征分类为密集特征、稀疏特征和序列特征
dense_features_list = [col for col in df.columns if 'dense' in col]
sparse_features_list = [col for col in df.columns if 'sparse' in col] + ['user_id', 'item_id']
sequence_features_list = [col for col in df.columns if 'sequence' in col]

现在我们开始准备模型，我们需要将模型需要的不同特征进行定义，并传给模型，这里需要用到nextrec内置的三种特征DenseFeature, SequenceFeature, SparseFeature。

In [ ]:
from nextrec.basic.features import DenseFeature, SequenceFeature, SparseFeature

# 我们将所有的数值特征作为DenseFeature，proj_dim=1表示不进行投影，当proj_dim大于1时表示对数值特征进行线性变换，类似于embedding的效果
dense_features = [DenseFeature(name=feat, proj_dim=1) for feat in dense_features_list] 

# 稀疏特征和序列特征我们一般会进行embedding，embedding_dim可以根据实际情况调整
# vocab_size 需要覆盖数据里的最大取值（从0开始），避免 embedding 索引越界
sparse_features = []
for feat in sparse_features_list:
    vocab_size = 20001
    # sparsefeature还可以设置一些其他的参数，例如初始化器，正则化和embedding_name等，当两个特征共享embedding时，可以设置相同的embedding_name       
    sparse_features.append(SparseFeature(name=feat, vocab_size=vocab_size, embedding_dim=4, embedding_name=feat)) 

# 序列特征的处理和稀疏特征类似，不过还需要设置序列的最大长度max_len和padding_idx等参数
sequence_features = []
for feat in sequence_features_list:
    vocab_size = 500
    sequence_features.append(
        SequenceFeature(
            name=feat,
            vocab_size=vocab_size,
            max_len=20,
            embedding_dim=8,
            padding_idx=0
        )
    )


在定义完特征后，我们来选择想要训练的模型。NextRec提供了超过20种工业界常用的召回，精排，多任务模型。这里我们先用经典的MMOE训练一个模型。

在此之前，我们需要实例化模型，并且为模型分配参数，优化器，调度器和损失函数。NextRec支持超过8种优化器，10种调度器和20种损失函数以及不平衡损失。

In [ ]:
from nextrec.models.multitask.mmoe import MMOE

# mmoe需要设置专家网络和任务塔的参数，这里我们设置4个专家网络，每个专家网络包含两层，任务塔也包含两层
# 我们拥有两个任务，分别是click和conversion，每个任务都是二分类任务，因此task参数设置为['binary', 'binary']
model = MMOE(
    dense_features=dense_features,
    sparse_features=sparse_features,
    sequence_features=sequence_features,
    expert_mlp_params= {"hidden_dims": [128, 64],  "activation": "leaky_relu", "dropout": 0.3},
    num_experts=4,  # 4个专家网络
    tower_mlp_params_list=[{"hidden_dims": [64, 32], "activation": "leaky_relu", "dropout": 0.2},  # click任务
                       {"hidden_dims": [64, 32], "activation": "leaky_relu", "dropout": 0.2},  # conversion任务
                        ],
    target=task_labels,  # 多任务标签
    task=['binary', 'binary'],  # 每个任务类型
    device='cpu',
    embedding_l1_reg=1e-6,
    embedding_l2_reg=1e-5,
    dense_l1_reg=1e-5,
    dense_l2_reg=1e-4,
    session_id="mmoe_task"    # session id用于区分不同的训练任务，会将训练日志，checkpoint，模型参数等保存在以session_id命名的文件夹中
)

# 编译模型用于设置优化器和损失函数，统一在 compile 中配置。
# 这里我们使用adam优化器，学习率为1e-3，权重衰减为1e-5
# 每个任务的损失函数我们都使用二分类交叉熵损失函数
model.compile(
    optimizer="adam",
    optimizer_params={"lr": 1e-3, "weight_decay": 1e-5},
    loss=['bce', 'bce'],  # 每个任务的损失函数
 )

# 这里我们设置训练1个epoch，你可以根据实际情况调整
# 我们可以为每个任务设置评估指标，这里我们为每个任务都设置了AUC, Recall和Precision指标
# 注意你可以在nextrec_logs/中查看训练日志和模型checkpoint文件
# 通过设置use_tensorboard，use_wandb，use_swanlab参数以及wandb_kwargs，swanlab_kwargs，可以将训练日志同步到对应的平台上。
# 例如：use_swanlab=True, swanlab_kwargs={"project": "NextRec", "name": "MMOE_experiment"}
model.fit(
    train_data=df,
    metrics={
        'click': ['auc', 'recall', 'precision'],
        'conversion': ['auc', 'recall', 'precision']
    },
    epochs=1,
    valid_split=0.2 # 从训练数据中划分20%作为验证集
)

你可以通过设置`fit`方法里的`train_data`, `valid_data`, `valid_split`参数来调整模型训练中的训练/验证集比例，`train_data`和`valid_data`支持传入dataframe，dict，文件路径。

- 当传入`valid_data`时，将其作为验证集
- 当传入`train_data`和`valid_split`时，将会按比例划分出验证集

In [ ]:
from sklearn.model_selection import train_test_split

# 划分训练集和验证集
train_df, valid_df = train_test_split(df, test_size=0.2, random_state=2025)

model.fit(
    train_data=train_df,
    valid_data=valid_df,
    metrics={
        'click': ['auc', 'recall', 'precision'],
        'conversion': ['auc', 'recall', 'precision']
    },
    epochs=1,
)

现在，我们要用到DataLoader了，DataLoader通常为模型准备不断迭代的批次的数据。我们准备了RecDataLoader，它用起来不会很复杂。RecDataLoader支持传入dict，DataFrame，DataLoader以及路径，并且支持通过streaming=True来配置流式加载数据。

当然，RecDataLoader不是必须的。NextRec支持在不设置DataLoader的情况下进行训练，稍后你就能看到。

现在我们再来训练一个精排模型，这里我们训练一个AutoINT作为示例，任务目标从多目标变为单目标。

In [ ]:
from nextrec.models.ranking.autoint import AutoInt

target = 'conversion'

model = AutoInt(
    dense_features=dense_features,
    sparse_features=sparse_features,
    sequence_features=sequence_features,
    att_layer_num=3,
    att_embedding_dim=8,
    att_head_num=2,
    att_dropout=0.0,
    att_use_residual=True,
    target=target,
    device='cpu',
    embedding_l1_reg=1e-6,
    dense_l1_reg=1e-5,
    embedding_l2_reg=1e-5,
    dense_l2_reg=1e-4,
    session_id="autoint_task"
)

# 编译模型
model.compile(
    optimizer="adam",
    optimizer_params={
        "lr": 1e-3,
        "weight_decay": 1e-5
    },
    loss="bce",
)

# 训练模型
model.fit(
    train_data=df,
    valid_split=0.2,
    metrics=['auc',
             'recall',
             'precision'],
    epochs=1,
    batch_size=512,
    shuffle=True
)

下面是已经支持的模型，欢迎调用来测试效果

### 排序模型

| 模型 | 论文 | 年份 | 状态 |
|------|------|------|------|
| **FM** | Factorization Machines | ICDM 2010 | 已支持 |
| **AFM** | Attentional Factorization Machines: Learning the Weight of Feature Interactions via Attention Networks | IJCAI 2017 | 已支持 |
| **DeepFM** | DeepFM: A Factorization-Machine based Neural Network for CTR Prediction | IJCAI 2017 | 已支持 |
| **Wide&Deep** | Wide & Deep Learning for Recommender Systems | DLRS 2016 | 已支持 |
| **xDeepFM** | xDeepFM: Combining Explicit and Implicit Feature Interactions | KDD 2018 | 已支持 |
| **FiBiNET** | FiBiNET: Combining Feature Importance and Bilinear Feature Interaction for CTR Prediction | RecSys 2019 | 已支持 |
| **PNN** | Product-based Neural Networks for User Response Prediction | ICDM 2016 | 已支持 |
| **AutoInt** | AutoInt: Automatic Feature Interaction Learning | CIKM 2019 | 已支持 |
| **DCN** | Deep & Cross Network for Ad Click Predictions | ADKDD 2017 | 已支持 |
| **DIN** | Deep Interest Network for Click-Through Rate Prediction | KDD 2018 | 已支持 |
| **DIEN** | Deep Interest Evolution Network for Click-Through Rate Prediction | AAAI 2019 | 已支持 |
| **MaskNet** | MaskNet: Introducing Feature-wise Gating Blocks for High-dimensional Sparse Recommendation Data | 2020 | 已支持 |

### 召回模型

| 模型 | 论文 | 年份 | 状态 |
|------|------|------|------|
| **DSSM** | Learning Deep Structured Semantic Models | CIKM 2013 | 已支持 |
| **DSSM v2** | DSSM with pairwise BPR-style optimization | - | 已支持 |
| **YouTube DNN** | Deep Neural Networks for YouTube Recommendations | RecSys 2016 | 已支持 |
| **MIND** | Multi-Interest Network with Dynamic Routing | CIKM 2019 | 已支持 |
| **SDM** | Sequential Deep Matching Model | - | 已支持 |

### 多任务模型

| 模型 | 论文 | 年份 | 状态 |
|------|------|------|------|
| **MMOE** | Modeling Task Relationships in Multi-task Learning | KDD 2018 | 已支持 |
| **PLE** | Progressive Layered Extraction | RecSys 2020 | 已支持 |
| **ESMM** | Entire Space Multi-Task Model | SIGIR 2018 | 已支持 |
| **ShareBottom** | Multitask Learning | - | 已支持 |